In [2]:
words=open("names.csv",encoding="utf-8").read().splitlines()

In [3]:
import re
import torch

clean_words = []

for w in words:

    w = re.sub(r"[^A-Za-z\s'& -]", "", w)
    w = " ".join(w.split())
    if w:
        clean_words.append(w)

words = clean_words
words[:8]

['lavera',
 'fairytabs',
 'arkive',
 'gaa',
 'cultiv cosmetique',
 'the green recipe',
 'mlanine',
 'base cosmetics']

In [4]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)

{1: ' ', 2: '&', 3: "'", 4: '-', 5: 'a', 6: 'b', 7: 'c', 8: 'd', 9: 'e', 10: 'f', 11: 'g', 12: 'h', 13: 'i', 14: 'j', 15: 'k', 16: 'l', 17: 'm', 18: 'n', 19: 'o', 20: 'p', 21: 'q', 22: 'r', 23: 's', 24: 't', 25: 'u', 26: 'v', 27: 'w', 28: 'x', 29: 'y', 30: 'z', 0: '.'}


In [5]:
#dataset creation

X,Y=[],[]
block_size=3 #context length i.e how many char to know to predict next one

for w in words[:1]:
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context),'--->',itos[ix])
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)
        


lavera
... ---> l
..l ---> a
.la ---> v
lav ---> e
ave ---> r
ver ---> a
era ---> .


In [6]:
 #lookup table
C=torch.randn((31,2))
C[:5]

tensor([[ 1.5384, -0.0762],
        [ 0.2805, -0.4138],
        [ 0.8762,  0.9127],
        [ 1.1232,  1.0949],
        [-0.6221, -0.3320]])

In [7]:
emb=C[X]
emb[1]

tensor([[ 1.5384, -0.0762],
        [ 1.5384, -0.0762],
        [ 0.2512, -0.4192]])

In [8]:
emb.size()

torch.Size([7, 3, 2])

In [9]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1) #have to change is context len chnages therfore we use unbind

tensor([[ 1.5384, -0.0762,  1.5384, -0.0762,  1.5384, -0.0762],
        [ 1.5384, -0.0762,  1.5384, -0.0762,  0.2512, -0.4192],
        [ 1.5384, -0.0762,  0.2512, -0.4192, -0.0458, -1.5507],
        [ 0.2512, -0.4192, -0.0458, -1.5507, -1.0626, -0.8377],
        [-0.0458, -1.5507, -1.0626, -0.8377,  0.2890,  0.4670],
        [-1.0626, -0.8377,  0.2890,  0.4670, -0.6901, -0.9610],
        [ 0.2890,  0.4670, -0.6901, -0.9610, -0.0458, -1.5507]])

In [10]:
torch.cat(torch.unbind(emb,1),1) # even though unbind creates a view but cat copies and allocate new memomry therefore we switch to view

tensor([[ 1.5384, -0.0762,  1.5384, -0.0762,  1.5384, -0.0762],
        [ 1.5384, -0.0762,  1.5384, -0.0762,  0.2512, -0.4192],
        [ 1.5384, -0.0762,  0.2512, -0.4192, -0.0458, -1.5507],
        [ 0.2512, -0.4192, -0.0458, -1.5507, -1.0626, -0.8377],
        [-0.0458, -1.5507, -1.0626, -0.8377,  0.2890,  0.4670],
        [-1.0626, -0.8377,  0.2890,  0.4670, -0.6901, -0.9610],
        [ 0.2890,  0.4670, -0.6901, -0.9610, -0.0458, -1.5507]])

In [11]:
emb.view(7,6)

tensor([[ 1.5384, -0.0762,  1.5384, -0.0762,  1.5384, -0.0762],
        [ 1.5384, -0.0762,  1.5384, -0.0762,  0.2512, -0.4192],
        [ 1.5384, -0.0762,  0.2512, -0.4192, -0.0458, -1.5507],
        [ 0.2512, -0.4192, -0.0458, -1.5507, -1.0626, -0.8377],
        [-0.0458, -1.5507, -1.0626, -0.8377,  0.2890,  0.4670],
        [-1.0626, -0.8377,  0.2890,  0.4670, -0.6901, -0.9610],
        [ 0.2890,  0.4670, -0.6901, -0.9610, -0.0458, -1.5507]])

In [12]:
W1=torch.randn((6,100))
b1=torch.randn(100)

In [13]:

h=torch.tanh(emb.view(-1,6)@W1+b1) #check broadcasting rules
h

tensor([[ 0.3449,  0.9123, -0.9993,  0.9443,  0.9006,  0.9784,  0.9172, -0.4911,
         -1.0000,  0.9795, -0.9977,  0.8500, -0.1176, -0.8664,  1.0000,  0.8351,
         -0.4716,  0.8471,  0.9382,  0.9909,  0.9920, -0.9835,  0.9727,  0.9797,
         -0.5141,  0.9926,  0.8250,  0.9994,  0.9964, -0.0528, -0.9978, -0.9220,
          0.9660, -0.8495, -0.9340, -0.9885, -0.1346,  0.9993,  0.5818, -0.9270,
         -0.8099,  0.9129, -0.9788, -0.9192, -0.8526,  0.3609, -0.9989, -0.9274,
          0.5112,  0.4274, -0.9988,  0.9970, -0.0505, -1.0000,  1.0000,  0.8162,
         -0.9900,  0.5319,  0.9508,  0.9846, -0.6333, -0.8037, -0.1077,  0.8372,
          0.7724, -0.9906, -0.9898,  0.7469,  0.9945,  0.9998,  0.5502,  0.5723,
          0.9751, -0.6866,  0.4627, -0.7328,  0.9943, -0.9991,  0.3909,  0.9989,
         -0.5803,  0.9789,  0.4822,  0.4614, -0.6347, -0.9456,  0.9827, -0.9026,
          0.5505, -0.7367,  0.9602,  0.8492, -0.9876,  0.1091, -0.9983, -0.4737,
         -0.7359, -0.4344, -

In [14]:
#last layer
W2=torch.randn((100,31))
b2=torch.randn(31)

In [15]:
logits=h@W2+b2
counts=logits.exp()
probs=counts/counts.sum(1,keepdims=True)

In [16]:
probs

tensor([[3.0743e-08, 6.5199e-05, 2.4870e-10, 6.4433e-11, 1.9826e-14, 2.2156e-10,
         9.8748e-01, 4.2280e-05, 7.4055e-15, 2.6643e-07, 4.8826e-12, 9.9554e-09,
         8.0345e-15, 1.7634e-08, 1.6419e-07, 3.1399e-13, 2.2165e-08, 1.0823e-18,
         4.2056e-12, 2.3142e-06, 2.0254e-13, 6.3926e-10, 3.6592e-10, 5.9469e-12,
         2.1462e-12, 2.0659e-13, 1.2414e-02, 2.9092e-14, 3.4331e-10, 5.7965e-12,
         1.7368e-13],
        [1.4993e-06, 4.2349e-04, 2.2677e-04, 4.1250e-11, 6.9052e-12, 3.2070e-09,
         4.2300e-02, 1.2196e-06, 1.8145e-13, 1.0225e-06, 1.2496e-08, 1.3034e-06,
         5.1263e-15, 2.2966e-06, 5.9365e-05, 5.1947e-11, 2.5406e-07, 2.0965e-17,
         6.3486e-09, 3.4695e-06, 1.5452e-11, 1.2804e-09, 1.7179e-11, 7.0771e-11,
         4.8946e-11, 2.0350e-10, 9.5698e-01, 6.1148e-14, 4.7719e-07, 2.3822e-09,
         2.6624e-11],
        [1.9007e-06, 2.9031e-07, 7.8267e-05, 4.3923e-07, 6.5527e-11, 6.3036e-06,
         2.6294e-04, 1.4518e-09, 3.3944e-07, 2.5291e-04, 5.0604e-

In [17]:
loss=-probs[torch.arange(7),Y].log().mean()
loss

tensor(15.3061)

In [19]:
X.shape
Y.shape

torch.Size([7])

In [20]:
g=torch.Generator().manual_seed(2147483764)
C=torch.randn((31,2),generator=g)
W1=torch.randn((6,100),generator=g)
b1=torch.randn(100,generator=g)
W2=torch.randn((100,31),generator=g)
b2=torch.randn(31,generator=g)
parameters=[C,W1,b1,W2,b2]



In [ ]:
for p in parameters:
    p.requires_grad=True

In [ ]:
for _ in range(10):

    #minibatch 
    ix=torch.randint(0,X.shape[0],(32,))

    #forward pass
    emb=C[X[ix]]
    h=torch.tanh(emb.view(-1,6)@W1+b1)
    logits=h@W2+b2
    loss=F.cross_entropy(logits,Y[ix])
    print(loss.item())

    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()

    #update
    for p in parameters:
        p.data+=-0.1*p.grad

    